# Head Segmentation

**We will generate head segmentations using the [CHARM](https://www.sciencedirect.com/science/article/pii/S1053811920305309?via%3Dihub) pipeline implemented through the [SimNIBS](https://simnibs.github.io/simnibs/build/html/index.html) software. Please install SimNIBS following their [instructions](https://simnibs.github.io/simnibs/build/html/installation/simnibs_installer.html) before continuing.** <br>

To use Charm in a terminal, head to the folder containing the T1 (and T2 optional) and run the following command line: 
```bash
charm subID path/to/T1.nii.gz path/to/T2.nii.gz --segment --initatlas
```

The subject ID (subID) is given as first argument. Charm will create a folder named `m2m_subID` that contains the segmentation results. The input images are given as final arguments (first the T1, then the T2, where T2 is optional).

Alternatively, run the following cells which will run the command line for the whole dataset.

📋 **Requirements**:
- FreeSurfer reconstruction.


## 📦 Imports

In [1]:
from pathlib import Path
import subprocess
import mindmap as mm

⚡ **Utilities**

In [ ]:
def run_charm(subID:str, T1: str | Path, T2: str | Path=None):
    """
    Run the charm command line.

    Parameters
    ----------
    folder: Path | str
        Folder in which to generate the output.

    subID: str
        Subject ID.

    T1: Path | str
        Path to T1 image.

    T2: Path | str
        Path to T2 image.

    Files
    ------
    m2m_subID: folder
        Folder containing charm segmentation.
        - m2m_subId/segmentation/labeling.nii.gz: Segemented image.
        - m2m_subId/segmentation/labeling_LUT.txt: CLUT of image.
        - ...
    """
    cleanT1 = False 
    cleanT2 = False

    # check T1
    if isinstance(T1, str):
        T1 = Path(T1)

    if not T1.exists():
        raise FileNotFoundError("T1 not found")

    folder = T1.parent
    if T1.suffix == ".mgz":
        cleanT1  = True
        T1_input = T1.name.replace(".mgz", ".nii.gz")
        out      = subprocess.run(["mri_convert", T1.name, T1_input], cwd=str(folder))
        out.check_returncode()
        
    else:
        T1_input = T1.name

    # check T2
    if T2:
        if isinstance(T2, str):
            T2 = Path(T2)

        if not T2.exists():
            raise FileNotFoundError("T2 not found")

        folderT2 = T2.parent
        if folderT2 != folder:
            raise ValueError("Parent folder of T1 and T2 must be the same to run charm.")

        if T2.suffix == ".mgz":
            cleanT2  = True
            T2_input = T2.name.replace(".mgz", ".nii.gz")
            out      = subprocess.run(["mri_convert", T2.name, T2_input], cwd=str(folder)) 
            out.check_returncode()
        else:
            T2_input = T2.name

        cml = ["charm", subID, T1_input, T2_input, "--segment", "--initatlas"]  # command line with T2
    else:
        cml = ["charm", subID, T1_input, "--segment", "--initatlas"]            # command line without T2

    # run charm
    out = subprocess.run(cml, cwd=str(folder))

    # clean nii files if original files were mgz
    if cleanT1:
        T1_input = folder/T1_input 
        T1_input.unlink()

    if cleanT2:
        T2_input = folder/T2_input
        T2_input.unlink()

    return

def segment(config, subjects=None, overwrite=False):
    """
    Segment whole dataset using charm. 

    Parameters
    ----------
    config: dict
        Configuration
        - FS_DIR: FreeSurfer directory.
        - SUBS: List of subjects.

    subjects:
        List of subjects. (Has priority over config[SUBS])
    """
    try: 
        fs_dir   = config["FS_DIR"]
        subjects = config["SUBS"] if subjects is None else subjects
    except KeyError:
        raise ValueError("FreeSurfer directory (FS_DIR) and subject list (SUBS) are necessary to segment.")

    for sub in subjects:
        seg = fs_dir/sub/"mri"/f"m2m_{sub}"/"segmentation"/"labeling.nii.gz"
        if seg.exists() and not overwrite:
            print(f"\033[93mSubject {sub} already has a segmentation.\033[0m")
            continue

        T1 = fs_dir/sub/"mri"/"T1.mgz"
        
        print(f"\033[94mRunning charm segementation for subject {sub}\033[0m")
        try:
            run_charm(sub, T1)
        except Exception as e:
            print(f"\033[91mSegmentation failed for subject {sub} because: {e}\033[0m")
        else: 
            print(f"\033[92mSuccesffully segmented subject {sub}\033[0m")
    
    

#### ⚙️ **<font color=green>Run CHARM segmentation</font>**
📥 **<font color=pink>Inputs</font>**
- FREESURFER reconstruction folder (`FS_DIR`):
    - T1

📤 **<font color=pink>Outputs</font>**<br>
At `[FS_DIR]/[subID]/mri` you will find a new folder `m2m_[subID]/segmentation` containing:
- `labeling.nii.gz`: Head segmentation volume.
- `labeling_LUT.txt`: Segementation's color LUT.

🛠️ **<font color=violet>Configuration</font>**:
- `fconfig`: Configuration file.

In [ ]:
####################### Parameters ###########################
fconfig = "../configurations/spike.yml"
##############################################################

config = mm.io.read_config(fconfig)
segment(config)

[ simnibs.segmentation.simnibs_samseg.ProbabilisticAtlas ] INFO: maximalDeformation=0.2128 minLogLikelihood=-1814501.9849
[ simnibs.segmentation.simnibs_samseg.ProbabilisticAtlas ] INFO: maximalDeformation=0.1053 minLogLikelihood=-1814621.2879
[ simnibs.segmentation.simnibs_samseg.ProbabilisticAtlas ] INFO: maximalDeformation=0.2592 minLogLikelihood=-1814760.2531
[ simnibs.segmentation.simnibs_samseg.ProbabilisticAtlas ] INFO: maximalDeformation=0.2068 minLogLikelihood=-1814793.3426
[ simnibs.segmentation.simnibs_samseg.ProbabilisticAtlas ] INFO: maximalDeformation=0.1278 minLogLikelihood=-1814819.6282
[ simnibs.segmentation.simnibs_samseg.ProbabilisticAtlas ] INFO: maximalDeformation=0.0653 minLogLikelihood=-1814841.3014
[ simnibs.segmentation.simnibs_samseg.ProbabilisticAtlas ] INFO: maximalDeformation=0.0351 minLogLikelihood=-1814846.2942
[ simnibs.segmentation.simnibs_samseg.ProbabilisticAtlas ] INFO: maximalDeformation=0.0000 minLogLikelihood=-1814846.2942
[ simnibs.segmentation.s

1
0.398942
SLIDING
L-BFGS
BFGS-MaximumMemoryLength: 12
LineSearchMaximalDeformationIntervalStopCriterion: 0.001
MaximalDeformationStopCriterion: 0.001
MaximumNumberOfIterations: 20
Verbose: 0


[ simnibs.segmentation.simnibs_samseg.ProbabilisticAtlas ] INFO: maximalDeformation=0.1819 minLogLikelihood=-1814886.0006
[ simnibs.segmentation.simnibs_samseg.ProbabilisticAtlas ] INFO: maximalDeformation=0.0628 minLogLikelihood=-1814947.0650
[ simnibs.segmentation.simnibs_samseg.ProbabilisticAtlas ] INFO: maximalDeformation=0.4177 minLogLikelihood=-1815040.2498
[ simnibs.segmentation.simnibs_samseg.ProbabilisticAtlas ] INFO: maximalDeformation=0.0283 minLogLikelihood=-1815062.3187
[ simnibs.segmentation.simnibs_samseg.ProbabilisticAtlas ] INFO: maximalDeformation=0.0068 minLogLikelihood=-1815067.6369
[ simnibs.segmentation.simnibs_samseg.ProbabilisticAtlas ] INFO: maximalDeformation=0.0131 minLogLikelihood=-1815069.6121
[ simnibs.segmentation.simnibs_samseg.ProbabilisticAtlas ] INFO: maximalDeformation=0.0099 minLogLikelihood=-1815072.8340
[ simnibs.segmentation.simnibs_samseg.ProbabilisticAtlas ] INFO: maximalDeformation=0.0140 minLogLikelihood=-1815074.5980
[ simnibs.segmentation.s

Constructing image-to-world transform from header information (/media/nino/DATA/SPIKE/derivatives/freesurfer-7.4.1/sub-pat02/mri/m2m_sub-pat02/T1.nii.gz)
Read image: /media/nino/DATA/SPIKE/derivatives/freesurfer-7.4.1/sub-pat02/mri/m2m_sub-pat02/T1.nii.gz
Wrote image and transform to file /media/nino/DATA/SPIKE/derivatives/freesurfer-7.4.1/sub-pat02/mri/m2m_sub-pat02/segmentation/T1_bias_corrected.nii.gz
Read mesh collection: /media/nino/DATA/SPIKE/derivatives/freesurfer-7.4.1/sub-pat02/mri/m2m_sub-pat02/segmentation/atlas_level2.txt.gz
Transforming points
Transforming points
Read mesh collection: /media/nino/DATA/SPIKE/derivatives/freesurfer-7.4.1/sub-pat02/mri/m2m_sub-pat02/segmentation/atlas_level2.txt.gz
Read mesh collection: /media/nino/DATA/SPIKE/derivatives/freesurfer-7.4.1/sub-pat02/mri/m2m_sub-pat02/segmentation/atlas_level2.txt.gz
done
Segmenting, can take a while...
Segmented 1 out of 55 structures.
Segmented 2 out of 55 structures.
Segmented 3 out of 55 structures.
Segmente

[ simnibs ] INFO: Writing out MNI warps.


Read mesh collection: /media/nino/DATA/SPIKE/derivatives/freesurfer-7.4.1/sub-pat02/mri/m2m_sub-pat02/segmentation/atlas_level2.txt.gz
Transforming points
Transforming points
Read mesh collection: /media/nino/DATA/SPIKE/derivatives/freesurfer-7.4.1/sub-pat02/mri/m2m_sub-pat02/segmentation/atlas_level2.txt.gz
Read mesh collection: /media/nino/DATA/SPIKE/derivatives/freesurfer-7.4.1/sub-pat02/mri/m2m_sub-pat02/segmentation/atlas_level2.txt.gz
Constructing image-to-world transform from header information (/home/nino/SimNIBS-4.5/simnibs_env/lib/python3.11/site-packages/simnibs/segmentation/atlases/charm_atlas_mni/template.nii)
Read image: /home/nino/SimNIBS-4.5/simnibs_env/lib/python3.11/site-packages/simnibs/segmentation/atlases/charm_atlas_mni/template.nii
Read mesh collection: /media/nino/DATA/SPIKE/derivatives/freesurfer-7.4.1/sub-pat02/mri/m2m_sub-pat02/segmentation/atlas_level2.txt.gz
Read mesh collection: /media/nino/DATA/SPIKE/derivatives/freesurfer-7.4.1/sub-pat02/mri/m2m_sub-pat0

... (2,532 outputs left out)

[ simnibs ] INFO: Writing out MNI warps.


Read mesh collection: /media/nino/DATA/SPIKE/derivatives/freesurfer-7.4.1/sub-pat15/mri/m2m_sub-pat15/segmentation/atlas_level2.txt.gz
Transforming points
Transforming points
Read mesh collection: /media/nino/DATA/SPIKE/derivatives/freesurfer-7.4.1/sub-pat15/mri/m2m_sub-pat15/segmentation/atlas_level2.txt.gz
Read mesh collection: /media/nino/DATA/SPIKE/derivatives/freesurfer-7.4.1/sub-pat15/mri/m2m_sub-pat15/segmentation/atlas_level2.txt.gz
Constructing image-to-world transform from header information (/home/nino/SimNIBS-4.5/simnibs_env/lib/python3.11/site-packages/simnibs/segmentation/atlases/charm_atlas_mni/template.nii)
Read image: /home/nino/SimNIBS-4.5/simnibs_env/lib/python3.11/site-packages/simnibs/segmentation/atlases/charm_atlas_mni/template.nii
Read mesh collection: /media/nino/DATA/SPIKE/derivatives/freesurfer-7.4.1/sub-pat15/mri/m2m_sub-pat15/segmentation/atlas_level2.txt.gz
Read mesh collection: /media/nino/DATA/SPIKE/derivatives/freesurfer-7.4.1/sub-pat15/mri/m2m_sub-pat1

[ simnibs ] INFO: Post-processing segmentation
[ simnibs ] INFO: Upsampling bias corrected images.


['/media/nino/DATA/SPIKE/derivatives/freesurfer-7.4.1/sub-pat15/mri/m2m_sub-pat15/label_prep/T1_upsampled.nii.gz']
/media/nino/DATA/SPIKE/derivatives/freesurfer-7.4.1/sub-pat15/mri/m2m_sub-pat15/segmentation/template_coregistered.mgz
Constructing image-to-world transform from header information (/media/nino/DATA/SPIKE/derivatives/freesurfer-7.4.1/sub-pat15/mri/m2m_sub-pat15/label_prep/T1_upsampled.nii.gz)
Constructing image-to-world transform from header information (/media/nino/DATA/SPIKE/derivatives/freesurfer-7.4.1/sub-pat15/mri/m2m_sub-pat15/segmentation/template_coregistered.mgz)
Read image: /media/nino/DATA/SPIKE/derivatives/freesurfer-7.4.1/sub-pat15/mri/m2m_sub-pat15/label_prep/T1_upsampled.nii.gz cropped by /media/nino/DATA/SPIKE/derivatives/freesurfer-7.4.1/sub-pat15/mri/m2m_sub-pat15/segmentation/template_coregistered.mgz
Constructing image-to-world transform from header information (/media/nino/DATA/SPIKE/derivatives/freesurfer-7.4.1/sub-pat15/mri/m2m_sub-pat15/label_prep/T